In [2]:
import math
from typing import Generator

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
from tqdm import tqdm

from zoo.utils import get_device

In [3]:
dtype = torch.float16
device = get_device()
print(f"Using dtype: {dtype}")
print(f"Using device: {device}")

Using dtype: torch.float16
Using device: mps


In [ ]:
class Tokenizer:
    def __init__(self, dataset: str):
        # CHANGE: sorted() — set iteration order is non-deterministic across runs,
        # so vocab indices would change every time, making saved models unloadable.
        self.vocab = sorted(set(dataset))
        self.word2idx = {ch: idx for idx, ch in enumerate(self.vocab)}
        self.idx2word = {idx: ch for idx, ch in enumerate(self.vocab)}

    def __len__(self):
        return len(self.vocab)

    def encode(self, text: str) -> list[int]:
        return [self.word2idx[ch] for ch in text]

    def decode(self, indices: list[int]) -> str:
        return "".join(self.idx2word[idx] for idx in indices)


with open("../data/tinyshakespeare.txt", "r") as f:
    dataset = f.read()

# CHANGE: fit tokenizer on FULL dataset before splitting —
# otherwise test_data may contain characters unseen in train vocab,
# causing KeyError in encode().
tokenizer = Tokenizer(dataset)
vocab_size = len(tokenizer)

split = int(len(dataset) * 0.8)
train_data = torch.tensor(tokenizer.encode(dataset[:split]), dtype=torch.long)
test_data = torch.tensor(tokenizer.encode(dataset[split:]), dtype=torch.long)

# CHANGE: keep data on CPU, move batches to device in the loop.
# Putting the entire dataset on MPS eats device memory and prevents
# pin_memory + non_blocking async transfers from working.
block_size = 128


def to_blocks(data: torch.Tensor, block_size: int) -> torch.Tensor:
    # Trim to exact multiple, then reshape — same as before but extracted to fn
    n = (len(data) // block_size) * block_size
    return data[:n].view(-1, block_size)  # (num_blocks, block_size) — stays on CPU


train_data = to_blocks(train_data, block_size)
test_data = to_blocks(test_data, block_size)


def batches(
    data: torch.Tensor,
    batch_size: int,
) -> Generator[tuple[torch.Tensor, torch.Tensor], None, None]:
    # CHANGE: shuffle indices each epoch so the model doesn't memorise block order.
    indices = torch.randperm(len(data) - 1)  # -1 so x[i]+1 never goes out of bounds
    for i in range(0, len(indices) - batch_size, batch_size):
        batch_idx = indices[i : i + batch_size]
        x = data[batch_idx]  # (B, block_size)
        y = data[batch_idx + 1]  # CHANGE: was data[i+1 : i+1+batch_size] which
        yield x, y  # sliced ROWS not shifted columns — y should be
        # the next-token target, i.e. the NEXT block row,
        # not x offset by one character.

In [ ]:
# ── RoPE ──────────────────────────────────────────────────────────────────────
# CHANGE: pre-cache cos/sin up to max_seq_length at init time instead of
#         recomputing tensors (arange, einsum, cat, cos, sin) on every forward.
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_length=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_length)

    def _build_cache(self, seq_len):
        t = torch.arange(
            seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device
        )  # type: ignore
        freqs = torch.outer(
            t, self.inv_freq
        )  # (S, head_dim/2) — outer is faster than einsum
        emb = torch.cat([freqs, freqs], dim=-1)  # (S, head_dim)
        # persistent=False: not saved in state_dict, rebuilt on load
        self.register_buffer("cos_cached", emb.cos()[None, None], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None], persistent=False)

    def forward(self, seq_len: int):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]


# CHANGE: standard RoPE rotation — [-x2, x1] form lets us write
#         q*cos + rotate_half(q)*sin in two fused ops instead of four sliced ones.
def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    return (q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin)


# ── Attention ─────────────────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads, max_seq_length=2048):
        super().__init__()
        assert embed_size % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads

        # CHANGE: fused QKV — one matmul instead of three, better memory locality
        # CHANGE: bias=False — modern LLMs drop biases; saves ~2% params & bandwidth
        self.qkv = nn.Linear(embed_size, 3 * embed_size, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size, bias=False)
        self.rope = RotaryPositionalEmbedding(self.head_dim, max_seq_length)

    def forward(self, x):
        B, S, E = x.shape

        # Single projection then split — one kernel launch instead of three
        Q, K, V = self.qkv(x).view(B, S, 3, self.num_heads, self.head_dim).unbind(2)
        Q = Q.transpose(1, 2)  # (B, H, S, D)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        cos, sin = self.rope(S)
        Q, K = apply_rope(Q, K, cos, sin)

        # CHANGE: F.scaled_dot_product_attention — uses Flash Attention on MPS/CUDA,
        #         fuses softmax+dropout+matmul, avoids materialising the full (S,S) matrix.
        #         is_causal=True generates the causal mask internally at near-zero cost.
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        return self.fc_out(out.transpose(1, 2).contiguous().view(B, S, E))


# ── Norms ─────────────────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(
        self, embed_size, eps=1e-6
    ):  # CHANGE: 1e-6 is standard (1e-8 underflows fp16)
        super().__init__()
        self.weight = nn.Parameter(torch.ones(embed_size))
        self.eps = eps

    def forward(self, x):
        # CHANGE: rsqrt instead of sqrt+div — single instruction on all hardware
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


# ── FFN ───────────────────────────────────────────────────────────────────────
# CHANGE: SwiGLU replaces plain GeLU MLP. Same param count (roughly) but the
#         gated path consistently improves loss at equivalent compute (LLaMA, PaLM).
#         Formula: down( silu(gate(x)) * up(x) )
class MLP(nn.Module):
    def __init__(self, embed_size, expansion_factor=4):
        super().__init__()
        hidden = int(embed_size * expansion_factor * 2 / 3)  # compensate for extra proj
        hidden = (hidden + 7) // 8 * 8  # round to multiple of 8 for tensor cores
        self.gate = nn.Linear(embed_size, hidden, bias=False)
        self.up = nn.Linear(embed_size, hidden, bias=False)
        self.down = nn.Linear(hidden, embed_size, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


# ── MoE ───────────────────────────────────────────────────────────────────────
class MixtureOfExperts(nn.Module):
    def __init__(self, embed_size, num_experts=4, top_k=2, expansion_factor=4):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        hidden = int(embed_size * expansion_factor * 2 / 3)
        hidden = (hidden + 7) // 8 * 8

        self.w_gate = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_up = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_down = nn.Parameter(torch.empty(num_experts, hidden, embed_size))
        self.router = nn.Linear(embed_size, num_experts, bias=False)

        for w in (self.w_gate, self.w_up, self.w_down):
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))

    def forward(self, x):
        B, S, E = x.shape
        T = B * S
        x_flat = x.view(T, E)

        # Router with top-k selection
        logits = self.router(x_flat)  # (T, E)
        topk_w, topk_ids = torch.topk(logits, self.top_k, dim=-1)  # (T, k)
        topk_w = F.softmax(topk_w, dim=-1)  # (T, k)

        # CHANGE: sparse gate matrix — scatter top-k weights, zeros elsewhere.
        #         Multiplying by this mask before einsum means non-selected experts
        #         contribute zero and get skipped in the weighted sum.
        gate = torch.zeros(T, self.num_experts, device=x.device, dtype=x.dtype)
        gate.scatter_(1, topk_ids, topk_w)  # (T, num_experts)

        # CHANGE: SwiGLU inside MoE — gate and up projected in one batched einsum each
        h_gate = torch.einsum("te,neh->tnh", x_flat, self.w_gate)  # (T, N, H)
        h_up = torch.einsum("te,neh->tnh", x_flat, self.w_up)  # (T, N, H)
        h = F.silu(h_gate) * h_up  # (T, N, H)

        # Weight by gate scores before down-projection — fuses weighting into the contraction
        h = h * gate.unsqueeze(-1)  # (T, N, H)
        out = torch.einsum("tnh,nhe->te", h, self.w_down)  # (T, E)

        return out.view(B, S, E)


# ── Block ─────────────────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, embed_size, num_heads, block_num):
        super().__init__()
        self.attn = MultiHeadAttention(embed_size, num_heads)
        self.ffn = MixtureOfExperts(embed_size)
        self.norm1 = RMSNorm(embed_size)
        self.norm2 = RMSNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # pre-norm
        x = x + self.ffn(self.norm2(x))
        return x


# ── LLM ───────────────────────────────────────────────────────────────────────
class LLM(nn.Module):
    def __init__(self, vocab_size, embed_size, num_blocks, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.blocks = nn.ModuleList(
            [Block(embed_size, num_heads, i) for i in range(num_blocks)]
        )
        # CHANGE: final RMSNorm before output head — stabilises logit scale
        self.norm_out = RMSNorm(embed_size)
        self.output_layer = nn.Linear(embed_size, vocab_size, bias=False)

        # CHANGE: weight tying — output projection shares weights with embedding.
        #         Halves the largest param tensor, improves generalisation (GPT-2, LLaMA).
        self.output_layer.weight = self.embedding.weight

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.norm_out(x))


model = LLM(vocab_size, embed_size=64, num_blocks=8).to(device, dtype=dtype)

model = torch.compile(model, backend="aot_eager")

summary(model, input_size=(1, 10), dtypes=[torch.long], device=device)

Layer (type:depth-idx)                                       Output Shape              Param #
OptimizedModule                                              [1, 10, 65]               --
├─LLM: 1-1                                                   [1, 10, 65]               --
│    └─Embedding: 2-1                                        [1, 10, 64]               4,160
│    └─ModuleList: 2-2                                       --                        --
│    │    └─Block: 3-1                                       [1, 10, 64]               151,936
│    │    └─Block: 3-2                                       [1, 10, 64]               151,936
│    │    └─Block: 3-3                                       [1, 10, 64]               151,936
│    │    └─Block: 3-4                                       [1, 10, 64]               151,936
│    │    └─Block: 3-5                                       [1, 10, 64]               151,936
│    │    └─Block: 3-6                                       [1, 10

In [11]:
import time


@torch.inference_mode()
def generate(model, tokenizer, prompt, max_length=20, temperature=1.0, verbose=True):
    model.eval()
    input_ids = torch.tensor(
        tokenizer.encode(prompt), dtype=torch.long, device=device
    ).unsqueeze(0)

    start_time = time.time()
    for _ in range(max_length - len(input_ids[0])):
        output = model(input_ids)
        next_token_logits = output[:, -1, :] / temperature
        next_token = torch.multinomial(
            F.softmax(next_token_logits, dim=-1), num_samples=1
        )
        input_ids = torch.cat([input_ids, next_token], dim=1)
    end_time = time.time()

    output = tokenizer.decode(input_ids.squeeze().tolist())

    if verbose:
        print(f"Total Time: {end_time - start_time:.4f} seconds")
        print(f"Tokens per second: {input_ids.size(1) / (end_time - start_time):.2f}")

    return output


print(generate(model, tokenizer, max_length=128, prompt="MIRANDA:"))

Total Time: 4.1854 seconds
Tokens per second: 30.58
MIRANDA:::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=1000
)

In [14]:
# Training loop
for epoch in range(10):
    model.train()
    pbar = tqdm(
        batches(train_data, batch_size=128),
        total=(len(train_data) - 1) // 128,
        desc=f"Epoch {epoch + 1}",
    )

    for x, y in pbar:
        # CHANGE: non_blocking async transfer now works because data is on CPU
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("mps", dtype=torch.float16):
            loss = criterion(model(x).view(-1, vocab_size), y.view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}"
        )

    print(generate(model, tokenizer, max_length=128, prompt="MIRANDA:"))

Epoch 1:  39%|███▉      | 21/54 [00:09<00:14,  2.28it/s, loss=nan, lr=3.9e-04]


KeyboardInterrupt: 

In [ ]:
gradients = []
for name, param in model.named_parameters():
    if param.grad is not None:
        weight_norm = param.data.norm().item()
        grad_norm = param.grad.norm().item()
        ratio = grad_norm / (param.data.norm().item() + 1e-8)  # Avoid division by zero
        gradients.append(
            {
                "name": name,
                "weight_norm": weight_norm,
                "grad_norm": grad_norm,
                "ratio": ratio,
            }
        )

# Show all
show_weights = False
show_biases = True

print(f"{'Name':<45} {'Weight Norm':<25} {'Grad Norm':<25} {'Ratio':<25}")
for grad in gradients:
    if (show_weights and "weight" in grad["name"]) or (
        show_biases and "bias" in grad["name"]
    ):
        print(
            f"{grad['name']:<45} {grad['weight_norm']:<25.4e} {grad['grad_norm']:<25.4e} {grad['ratio']:<25.4e}"
        )

In [ ]:
import time

import torch
from torch.profiler import ProfilerActivity, profile, record_function


# ── 1. Quick sanity — raw throughput without compile overhead ──────────────────
def benchmark_raw(model, vocab_size, device, dtype, seq_len=64, batch_size=4, steps=20):
    model.eval()
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    # Warmup (torch.compile traces on first call)
    with torch.no_grad():
        for _ in range(3):
            _ = model(x)
    if device == "mps":
        torch.mps.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(steps):
            _ = model(x)
    if device == "mps":
        torch.mps.synchronize()
    elapsed = time.perf_counter() - start

    tokens = batch_size * seq_len * steps
    print(
        f"[raw inference] {tokens / elapsed:.1f} tok/s  ({elapsed / steps * 1000:.1f} ms/step)"
    )


# ── 2. Training step breakdown — forward / backward / optimizer ────────────────
def benchmark_training_step(
    model, vocab_size, device, dtype, seq_len=64, batch_size=4, steps=20
):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    target = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    def sync():
        if device == "mps":
            torch.mps.synchronize()
        elif device == "cuda":
            torch.cuda.synchronize()

    # Warmup
    for _ in range(3):
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(
            model(x).view(-1, vocab_size), target.view(-1)
        )
        loss.backward()
        optimizer.step()
    sync()

    fwd_times, bwd_times, opt_times = [], [], []

    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)  # set_to_none=True: faster than zeroing

        # --- forward ---
        t0 = time.perf_counter()
        sync()
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, vocab_size), target.view(-1)
        )
        sync()
        fwd_times.append(time.perf_counter() - t0)

        # --- backward ---
        t1 = time.perf_counter()
        loss.backward()
        sync()
        bwd_times.append(time.perf_counter() - t1)

        # --- optimizer ---
        t2 = time.perf_counter()
        optimizer.step()
        sync()
        opt_times.append(time.perf_counter() - t2)

    def stats(name, times):
        t = torch.tensor(times) * 1000
        print(
            f"  {name:12s}  mean={t.mean():.1f}ms  std={t.std():.1f}ms  min={t.min():.1f}ms"
        )

    total = sum(fwd_times + bwd_times + opt_times)
    tok_per_s = (batch_size * seq_len * steps) / total
    print(f"\n[training breakdown]  {tok_per_s:.1f} tok/s total")
    stats("forward", fwd_times)
    stats("backward", bwd_times)
    stats("optimizer", opt_times)


# ── 3. Torch profiler — per-op breakdown with MPS activity ────────────────────
def profile_model(model, vocab_size, device, dtype, seq_len=64, batch_size=4):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    target = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    activities = [ProfilerActivity.CPU]
    if device == "cuda":
        activities.append(ProfilerActivity.CUDA)
    # MPS ops show up under CPU activity with "mps:" prefix

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        for _ in range(5):
            optimizer.zero_grad(set_to_none=True)
            with record_function("forward"):
                logits = model(x)
                loss = torch.nn.functional.cross_entropy(
                    logits.view(-1, vocab_size), target.view(-1)
                )
            with record_function("backward"):
                loss.backward()
            with record_function("optimizer"):
                optimizer.step()

    # Top ops by total CPU+device time
    print("\n[profiler] top 20 ops by self time:")
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))

    # Dump chrome trace — open at chrome://tracing or ui.perfetto.dev
    prof.export_chrome_trace("trace.json")
    print("\n[profiler] chrome trace saved → trace.json")
    print("           open at https://ui.perfetto.dev")


# ── 4. Isolated module timer — find the slow layer ────────────────────────────
def benchmark_modules(model, vocab_size, device, dtype, seq_len=64, batch_size=4):
    model.eval()
    results = {}

    def sync():
        if device == "mps":
            torch.mps.synchronize()
        elif device == "cuda":
            torch.cuda.synchronize()

    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    with torch.no_grad():
        emb = model.embedding(x)  # get realistic activations

    # Time each block individually
    for i, block in enumerate(model.blocks):
        inp = emb.clone()
        # warmup
        with torch.no_grad():
            for _ in range(3):
                _ = block(inp)
        sync()

        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(50):
                _ = block(inp)
        sync()
        ms = (time.perf_counter() - t0) / 50 * 1000
        results[f"block[{i}]"] = ms
        attn_type = "MHA" if block.attn is not None else "Identity"
        print(f"  block[{i}] ({attn_type} + MoE)  {ms:.2f}ms")

    slowest = max(results, key=results.get)
    print(f"\n  slowest: {slowest} at {results[slowest]:.2f}ms")


# ── 5. dtype / compile comparison ─────────────────────────────────────────────
def compare_configs(vocab_size, embed_size, num_blocks, device):
    from torchinfo import summary

    def run(use_compile, dtype):
        m = LLM(vocab_size, embed_size, num_blocks).to(device, dtype=dtype)
        if use_compile:
            try:
                m = torch.compile(m, mode="reduce-overhead")
            except Exception as e:
                print(f"  compile failed: {e}")
                return
        x = torch.randint(0, vocab_size, (4, 64), device=device)
        # warmup
        with torch.no_grad():
            for _ in range(5):
                m(x)
        if device == "mps":
            torch.mps.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(30):
                m(x)
        if device == "mps":
            torch.mps.synchronize()
        elapsed = time.perf_counter() - t0
        tok_s = 4 * 64 * 30 / elapsed
        label = f"compile={'on ' if use_compile else 'off'}  dtype={dtype}"
        print(f"  {label:45s}  {tok_s:7.1f} tok/s")

    print("\n[config comparison]")
    for compiled in [False, True]:
        for dt in [torch.float32, torch.float16, torch.bfloat16]:
            run(compiled, dt)


# ── Run everything ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(f"device={device}  dtype={dtype}")
    print("=" * 60)

    benchmark_raw(model, vocab_size, device, dtype)
    benchmark_training_step(model, vocab_size, device, dtype)
    benchmark_modules(model, vocab_size, device, dtype)
    profile_model(model, vocab_size, device, dtype)
    compare_configs(vocab_size, embed_size=64, num_blocks=8, device=device)